# Benchmark - comparison notebook
> Uses parquet artefacts generated by `inference.ipynb` to compare model performance.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import polars as pl
from fintl.common import Config
from PIL import Image, ImageFile
from pydantic import BaseModel
from fintl.etl.providers.scalable.broker20260309 import _get_ollama_client
import ipywidgets as widgets
from typing import cast, TypedDict
import instructor
import time
from openai.types.chat.chat_completion import ChatCompletion
from IPython.display import display, clear_output
from instructor.processing.multimodal import Image as InstructorImage
from fintl.etl.providers.scalable.broker20260309 import _BalanceInfoExtract, _SYSTEM_PROMPT, OllamaInferenceError
import tqdm
from plotnine import ggplot, aes, geom_point, geom_histogram
from polars._typing import PythonLiteral

from enum import StrEnum, auto

class ModelProvider(StrEnum):
    ollama = auto()
    llama_swap = auto()

In [ ]:
RESULTS_DIR = Path("./benchmark-results")
assert RESULTS_DIR.exists()
ollama_dir = RESULTS_DIR / ModelProvider.ollama.value
llama_swap_dir = RESULTS_DIR / ModelProvider.llama_swap.value
MODELS = {
    "ollama:qwen3.6:latest": ollama_dir / "qwen3.6-latest.parquet",
    "ollama:ministral-3:8b": ollama_dir / "ministral-3-8b.parquet",
    "ollama:ministral-3:14b": ollama_dir / "ministral-3-14b.parquet",
    "ollama:gemma4:latest": ollama_dir / "gemma4-latest.parquet",
    "llama-swap:qwen3.6-27b": llama_swap_dir / "qwen-3.6-27b.parquet",
    "llama-swap:gemma4-31b": llama_swap_dir / "gemma-4-31b.parquet",
}
for k,p in MODELS.items():
    assert p.exists(), f"{k}: {p=} does not exist"

## Load data

In [ ]:
benchmark_results = {
    k: pl.read_parquet(p) for k,p in MODELS.items()
}

## Analysis

Metrics:

- Completion rate: How many of the requests were successfully completed (independent of correctness)?
- Processing time (first run): How long each image took to process (independent of correctness)?
- Token usage (first run): How many tokens were used (independeny of correctness)?
- `amount` delta (all runs): How much does the prediction deviate from the ground trouth?
- `amount` stability: How many of the predictions per image were consistent across runs?

In [ ]:
def floatify(v: PythonLiteral | None) -> float:
    if v is None:
        msg = "Passed value was unexpectedly None."
        raise ValueError(msg)
    return float(v)

def calc_completion_rate(results_df:pl.DataFrame) -> tuple[int,int,float]:
    n_success = int(results_df["ok"].sum())
    n_total = len(results_df)
    print(f"Completed: {n_success:_} / {n_total:_}")
    return n_success, n_total, n_success / n_total

def calc_inference_stability(results_df:pl.DataFrame) -> float:

    case_consistencies = results_df.group_by("idx").agg((pl.col("y_pred").n_unique() == 1).alias("consistent")).sort("idx")
    overall_consistency = floatify(case_consistencies["consistent"].mean())
    print(f"Consistency: {overall_consistency:.2%}")
    return overall_consistency

def calc_processing_time(results_df:pl.DataFrame) -> tuple[float,float]:
    "First run processing time."
    d = results_df.filter(pl.col("run")==1)
    _mean = floatify(d["elapsed"].mean())
    _median = floatify(d["elapsed"].median())
    print(f"""Processing time:
    - mean: {_mean:.2f} s
    - median: {_median:.2f} s""")
    return _mean, _median
    
def calc_token_usage(results_df:pl.DataFrame) -> tuple[float,float]:
    
    _mean = floatify(results_df["total_tokens"].mean())
    _median = floatify(results_df["total_tokens"].median())
    print(f"""Token usage:
    - mean: {_mean:.0f}
    - median: {_median:.0f}""")
    return _mean, _median

def calc_amount_delta(results_df:pl.DataFrame) -> tuple[float,float]:

    d = results_df.filter(pl.col("ok") == True).with_columns(**{
        "delta": pl.col("y_pred") - pl.col("y_true")
    })
    _mean = floatify(d["delta"].mean())
    _median = floatify(d["delta"].median())
    print(f"""Amount delta:
    - mean: {_mean:.2f}
    - median: {_median:.2f}""")
    return _mean, _median

def calc_relative_amount_delta(results_df:pl.DataFrame) -> tuple[float,float]:

    d = results_df.filter(pl.col("ok") == True).with_columns(**{
        "delta": (pl.col("y_pred") - pl.col("y_true")) / pl.col("y_true")
    })
    _mean = floatify(d["delta"].mean())
    _median = floatify(d["delta"].median())
    print(f"""Relative amount delta:
    - mean: {_mean:.2%}
    - median: {_median:.2%}""")
    return _mean, _median

class ModelMetrics(TypedDict):
    model:str
    successfully_completed_requests:int
    total_requests:int
    successful_completion_rate:float
    overall_consistency:float
    mean_processing_time:float
    median_processing_Time:float
    mean_token_usage:float
    median_token_usage:float
    mean_amount_delta:float
    median_amount_delta:float
    mean_relative_amount_delta:float
    median_relative_amount_delta:float

stats: list[ModelMetrics] = []

for m, results_df in benchmark_results.items():
    
    print(f"\n\n========= {m} =========\n")
    successfully_completed_requests, total_requests, successful_completion_rate = calc_completion_rate(results_df)
    overall_consistency = calc_inference_stability(results_df)
    mean_processing_time, median_processing_Time = calc_processing_time(results_df)
    mean_token_usage, median_token_usage = calc_token_usage(results_df)
    mean_amount_delta, median_amount_delta = calc_amount_delta(results_df)
    mean_relative_amount_delta, median_relative_amount_delta = calc_relative_amount_delta(results_df)
        
    stats.append({
        "model": m,
        "successfully_completed_requests":successfully_completed_requests,
        "total_requests":total_requests,
        "successful_completion_rate":successful_completion_rate,
        "overall_consistency":overall_consistency,
        "mean_processing_time":mean_processing_time,
        "median_processing_Time":median_processing_Time,
        "mean_token_usage":mean_token_usage,
        "median_token_usage":median_token_usage,
        "mean_amount_delta":mean_amount_delta,
        "median_amount_delta":median_amount_delta,
        "mean_relative_amount_delta":mean_relative_amount_delta,
        "median_relative_amount_delta":median_relative_amount_delta
    })    

stats_df = pl.from_dicts(stats)
stats_df.head(6)

In [ ]:
stats_df.filter(pl.col("model").str.contains("3.6") | pl.col("model").str.contains("gemma")).select(["model","successful_completion_rate","overall_consistency","mean_processing_time","mean_token_usage","mean_amount_delta","mean_relative_amount_delta"])